# Chat with your data: semantic layer vs raw text-to-SQL

A local LLM answers business questions about the dbt demo shop in two ways:

1. **Semantic layer**: the [Kubeflow Semantic Operator](https://github.com/kubeflow/semantic-operator) serves an
   [Apache Ossie](https://github.com/apache/ossie) model of the shop (`dbt_demo`) over MCP. The LLM only *picks* certified
   metrics and dimensions (`list_metrics`, `list_dimensions`, `query_metric`); the server compiles **one governed SQL
   statement** for Trino. The LLM never writes SQL.
2. **Raw text-to-SQL** (the baseline): the same LLM gets the table DDL and writes Trino SQL itself.

Both answers are checked against hand-written SQL. The model's definitions matter: **revenue excludes returned orders**,
a rule the raw path has no way to know.

Runs on the Helm chart with `semantic.enabled=true` (the workstation profile has it). Docker Compose is frozen and has no
semantic layer, so the first cell stops there with a message.

In [ ]:
import os, sys, pathlib
import pandas as pd
sys.path.append(str(pathlib.Path.cwd().parent))        # lab_utils.py lives one folder up
import lab_utils as lu

lu.require_semantic()                                   # stops on Docker Compose
lu.require_tool_model(lu.TOOL_MODEL)                    # qwen3:4b: Ollama tool calling
MODEL = os.environ.get("SEMANTIC_DEMO_MODEL", "dbt_demo")
print("semantic layer:", os.environ["SEMANTIC_MCP_URL"], "| model:", MODEL, "| LLM:", lu.TOOL_MODEL)

## 1. The data and the model

The tables come from the dbt project (`../dbt/lakehouse_demo`). Build them if they are not there yet; the operator
re-checks its model every 30 s and publishes it once the tables exist.

In [ ]:
tables = set(lu.trino("SHOW TABLES FROM iceberg.dbt_demo")["Table"]) if "dbt_demo" in set(lu.trino("SHOW SCHEMAS FROM iceberg")["Schema"]) else set()
if not {"fct_orders", "stg_customers", "dim_products"} <= tables:
    lu.dbt("build")
lu.wait_for_model(MODEL)
pd.DataFrame(lu.mcp_call("list_models", {})["models"])

In [ ]:
metrics = pd.DataFrame(lu.mcp_call("list_metrics", {"model": MODEL})["metrics"])
dims = pd.DataFrame(lu.mcp_call("list_dimensions", {"model": MODEL})["dimensions"])
display(metrics[[c for c in ("name", "description", "synonyms") if c in metrics]])
display(dims[[c for c in ("name", "description") if c in dims]])

## 2. One question, step by step

The LLM's tool calls, the SQL the server compiled, and the rows Trino returned:

In [ ]:
r = lu.ask_semantic("What is our total revenue by country?")
print("tool calls:", " -> ".join(r["tool_calls"]))
print("\nSQL compiled by the semantic layer:\n", r["sql"])
print("\nrows:", r["rows"])
print("\nanswer:", r["answer"])

## 3. Five questions, both paths, scored

Ground truth is hand-written SQL that applies the model's definitions (revenue and average order value exclude returned
orders). Small local models vary between runs; the score table is the point, not one answer.

In [ ]:
REV = "SUM(CASE WHEN o.status <> 'returned' THEN o.amount ELSE 0 END)"
FROM = ("FROM iceberg.dbt_demo.fct_orders o JOIN iceberg.dbt_demo.stg_customers c ON c.customer_id = o.customer_id "
        "JOIN iceberg.dbt_demo.dim_products p ON p.product_id = o.product_id")
QUESTIONS = {
    "What is total revenue by country?": f"SELECT c.country, {REV} {FROM} GROUP BY 1",
    "How many orders are there per product category?": f"SELECT p.category, COUNT(DISTINCT o.order_id) {FROM} GROUP BY 1",
    "How many units were sold per product?": f"SELECT p.product_name, SUM(o.quantity) {FROM} GROUP BY 1",
    "What is the average order value for each subscription plan?":
        f"SELECT c.plan, {REV} / COUNT(DISTINCT o.order_id) {FROM} GROUP BY 1",
    "How many customers placed orders in each country?": f"SELECT c.country, COUNT(DISTINCT o.customer_id) {FROM} GROUP BY 1",
}
RAW_TABLES = ["fct_orders", "stg_customers", "dim_products"]

results = []
for q, truth_sql in QUESTIONS.items():
    truth = lu.trino(truth_sql).values.tolist()
    sem, raw = lu.ask_semantic(q), lu.ask_raw_sql(q, RAW_TABLES)
    results.append({"question": q,
                    "semantic": "✓" if lu.rows_match(sem["rows"], truth) else "✗",
                    "raw SQL": "✓" if lu.rows_match(raw["rows"], truth) else ("error" if raw["error"] else "✗"),
                    "tool calls": " -> ".join(sem["tool_calls"]),
                    "semantic SQL": sem["sql"], "raw SQL text": raw["sql"]})
score = pd.DataFrame(results)
print(f"semantic layer: {(score['semantic'] == '✓').sum()}/{len(score)}   raw text-to-SQL: {(score['raw SQL'] == '✓').sum()}/{len(score)}")
score[["question", "semantic", "raw SQL", "tool calls"]]

In [ ]:
# The SQL behind each answer (compare how the raw path treats returned orders)
for row in results:
    print("Q:", row["question"])
    print("  semantic:", " ".join((row["semantic SQL"] or "-").split()))
    print("  raw     :", " ".join((row["raw SQL text"] or "-").split()), "\n")

## 4. Governance

The model's `marketing` role may use every metric but not the customers' e-mail address (`denyFields`). The server
refuses the request; it does not silently drop the filter.

In [ ]:
denied = lu.mcp_call("query_metric", {"model": MODEL, "metrics": ["order_count"],
                                      "filters": [{"field": "stg_customers.email", "op": "LIKE", "value": "%@example.com"}]},
                     role="marketing")
print("marketing ->", denied.get("error") or denied)
allowed = lu.mcp_call("query_metric", {"model": MODEL, "metrics": ["order_count"], "dimensions": ["stg_customers.country"]},
                      role="marketing")
pd.DataFrame(allowed["rows"], columns=allowed["columns"])